task 1

In [1]:
import random

random.seed(42) 
tickets = []

categories = ['Login', 'Login', 'Login', 'Billing', 'Tech Support'] 

for i in range(200):
    
    minutes = random.randint(5, 60)
    
    if random.random() < 0.1:
        minutes = None

    ticket = {
        'ticket_id': 1000 + i,
        'customer_id': random.randint(100, 150),
        'category': random.choice(categories),
        'resolution_minutes': minutes,
        'escalated': random.choice([True, False])
    }
    
    tickets.append(ticket)

print(len(tickets))
print(tickets[:3])

200
[{'ticket_id': 1000, 'customer_id': 147, 'category': 'Login', 'resolution_minutes': 45, 'escalated': True}, {'ticket_id': 1001, 'customer_id': 106, 'category': 'Tech Support', 'resolution_minutes': 19, 'escalated': True}, {'ticket_id': 1002, 'customer_id': 101, 'category': 'Login', 'resolution_minutes': 42, 'escalated': True}]


I used random.seed(42) to make sure the data is the same every time I run the code. To make it look real, I made 'Login Issue' appear more often than other categories. I also made about 10% of the resolution_minutes either None so I can practice cleaning them in the next tasks.

task 2

In [2]:
def validate_keys(data):
    bad_records = []
    required = ['ticket_id', 'customer_id', 'category', 'resolution_minutes', 'escalated']
    
    for row in data:
        if not all(key in row for key in required):
            bad_records.append(row)
            
    return bad_records


def validate_resolution(data):
    bad_records = []
    
    for row in data:
        value = row.get('resolution_minutes')
        
        if type(value) != int:
            bad_records.append(row)
            
    return bad_records


missing_keys_list = validate_keys(tickets)
invalid_res_list = validate_resolution(tickets)

print("Missing Keys Count: ",len(missing_keys_list))
print("Invalid Resolution Count:", len(invalid_res_list))

Missing Keys Count:  0
Invalid Resolution Count: 20


task 3

In [3]:
def clean_records(data):
    cleaned_list = []
    
    for ticket in data:
        res_time = ticket.get('resolution_minutes')
        if type(res_time) == int:
            
            new_ticket = ticket.copy()
            
            original_cat = ticket.get('category', '')
            # Kateqoriyanı təmizləyirik
            new_ticket['category'] = original_cat.strip().title()
            
            cleaned_list.append(new_ticket)
            
    return cleaned_list

cleaned_data = clean_records(tickets)

print("Original Count:", len(tickets))
print("Cleaned Count:", len(cleaned_data))

print("Some cleaned Records:")
for t in cleaned_data[:5]:
    print(t)

Original Count: 200
Cleaned Count: 180
Some cleaned Records:
{'ticket_id': 1000, 'customer_id': 147, 'category': 'Login', 'resolution_minutes': 45, 'escalated': True}
{'ticket_id': 1001, 'customer_id': 106, 'category': 'Tech Support', 'resolution_minutes': 19, 'escalated': True}
{'ticket_id': 1002, 'customer_id': 101, 'category': 'Login', 'resolution_minutes': 42, 'escalated': True}
{'ticket_id': 1003, 'customer_id': 101, 'category': 'Tech Support', 'resolution_minutes': 19, 'escalated': True}
{'ticket_id': 1004, 'customer_id': 134, 'category': 'Billing', 'resolution_minutes': 50, 'escalated': True}


I decided to drop records where resolution_minutes was missing or invalid because they cannot be used for calculations. I also used .strip().title() to normalize category names.

task 4

average resolution time per category

In [4]:
def get_avg_resolution(data):
    totals = {} 
    counts = {} 
    
    for t in data:
        cat = t['category']
        time = t['resolution_minutes']
        
        if cat not in totals:
            totals[cat] = 0
            counts[cat] = 0
            
        totals[cat] += time
        counts[cat] += 1

    results = {}
    for cat in totals:
        results[cat] = round(totals[cat] / counts[cat], 2)
        
    return results

avg_res = get_avg_resolution(cleaned_data)
print("Avg Resolution:", avg_res)

Avg Resolution: {'Login': 33.59, 'Tech Support': 32.24, 'Billing': 31.52}


Count of tickets per customer


In [5]:
def get_customer_counts(data):
    counts = {}
    for t in data:
        cust = t['customer_id']
        if cust in counts:
            counts[cust] += 1
        else:
            counts[cust] = 1
    return counts

cust_counts = get_customer_counts(cleaned_data)

Escalation rate overall and by category


In [6]:
def get_escalation_rates(data):
    total_esc = sum(1 for t in data if t['escalated'])
    overall_rate = total_esc / len(data)
    
    results = {'overall': round(overall_rate, 2)}
    
    cat_counts = {} 
    
    for t in data:
        cat = t['category']
        if cat not in cat_counts:
            cat_counts[cat] = [0, 0]
            
        cat_counts[cat][1] += 1           
        if t['escalated']:
            cat_counts[cat][0] += 1       
            
    for cat, val in cat_counts.items():
        results[cat] = round(val[0] / val[1], 2)
        
    return results

esc_rates = get_escalation_rates(cleaned_data)
print("Escalation Rates:", esc_rates)

Escalation Rates: {'overall': 0.5, 'Login': 0.49, 'Tech Support': 0.49, 'Billing': 0.55}


a small validation check: the sum of category counts matches the total number of cleaned records. 

In [7]:
total_from_counts = sum(cust_counts.values())
print("\nValidation Check:", total_from_counts == len(cleaned_data))
print("Calculated:", total_from_counts, "Actual:", len(cleaned_data))


Validation Check: True
Calculated: 180 Actual: 180


task 5

In [8]:
def create_final_report(data):
    report = {
        'average_time': get_avg_resolution(data),
        'customer_counts': get_customer_counts(data),
        'escalation_rates': get_escalation_rates(data)
    }
    return report

final_report = create_final_report(cleaned_data)

print("Average Times:", final_report['average_time'])
print("Escalation Rates:", final_report['escalation_rates'])
print("Total Active Customers:", len(final_report['customer_counts']))

Average Times: {'Login': 33.59, 'Tech Support': 32.24, 'Billing': 31.52}
Escalation Rates: {'overall': 0.5, 'Login': 0.49, 'Tech Support': 0.49, 'Billing': 0.55}
Total Active Customers: 51


I checked the final report and found something interesting. Even though I expected 'Tech Support' to take the longest time, the data shows that 'Login' issues actually have the highest average resolution time at 33.59 minutes. Also, the escalation rate is almost the same for everyone, around 50%. This report is really helpful because it shows us exactly where the support team is struggling the most.